## 0) Setup


In [2]:
import os
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

# Paths:
DATA_DIR = os.path.join('..', 'data')
OUT_DIR  = os.path.join('..', 'output')
os.makedirs(OUT_DIR, exist_ok=True)

NACC_MAIN_PATH = os.path.join(DATA_DIR, 'investigator_nacc72.csv')

USE_ONLY_V3 = False   # True => keep only FORMVER in {3, 3.2}

# Horizon sweep range
MIN_H = 3
MAX_H = 12  


## 1) Load data + construct visit_date
building `visit_date` from `VISITYR`, `VISITMO`, `VISITDAY` with sensible defaults.


In [3]:
df = pd.read_csv(NACC_MAIN_PATH, low_memory=False)
df['NACCID'] = df['NACCID'].astype(str)

def make_visit_date(df_: pd.DataFrame) -> pd.Series:
    y = pd.to_numeric(df_['VISITYR'], errors='coerce')
    m = pd.to_numeric(df_['VISITMO'], errors='coerce').fillna(6).astype(int).clip(1, 12)
    d = pd.to_numeric(df_['VISITDAY'], errors='coerce').fillna(15).astype(int).clip(1, 28)
    return pd.to_datetime(dict(year=y, month=m, day=d), errors='coerce')

df['visit_date'] = make_visit_date(df)
df = df.dropna(subset=['NACCID', 'visit_date']).copy()
df = df.sort_values(['NACCID', 'visit_date']).reset_index(drop=True)

if 'FORMVER' in df.columns:
    df['FORMVER'] = pd.to_numeric(df['FORMVER'], errors='coerce')

print('Rows:', len(df), 'Participants:', df['NACCID'].nunique())
df[['NACCID','visit_date'] + ([ 'FORMVER' ] if 'FORMVER' in df.columns else [])].head()

/var/folders/nq/q3v4dg_s4q78s4w9bbr8zsv40000gn/T/ipykernel_13180/1939311393.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['visit_date'] = make_visit_date(df)


Rows: 207454 Participants: 55268


,NACCID,visit_date,FORMVER
0,NACC000011,2006-04-17,1.0
1,NACC000011,2007-06-18,1.0
2,NACC000011,2008-06-03,2.0
3,NACC000011,2009-08-03,2.0
4,NACC000034,2015-07-16,3.0


## 2) Optional FORMVER filter


In [4]:
USE_ONLY_V3 = False 

if USE_ONLY_V3 and 'FORMVER' in df.columns:
    before = len(df)
    df = df[df['FORMVER'].isin([3, 3.2])].copy()
    df = df.sort_values(['NACCID', 'visit_date']).reset_index(drop=True)
    print('Filtered to FORMVER in {3,3.2}: rows', before, '->', len(df))
else:
    print('No FORMVER filtering applied.')

No FORMVER filtering applied.


## 3) Baseline selection
Baseline is the earliest visit where:
- cognitive status is non-demented (CN/MCI-ish), and
- DEMENTED==0 (if available)


In [5]:
import numpy as np
import pandas as pd

def first_row_per_group(df_: pd.DataFrame, group_col: str, sort_col: str) -> pd.DataFrame:
    return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()

# DEMENTED: 0/1
no_dem_mask = (df["DEMENTED"] == 0) if "DEMENTED" in df.columns else pd.Series(True, index=df.index)

MISSING_CODES = {-4, 9}

if "NACCUDSD" in df.columns:
    # Primary baseline definition: CN/MCI (not dementia)
    uds_non_dem = df["NACCUDSD"].isin([1, 2, 3])

    # Fallback for rows where NACCUDSD is missing or coded as somewhat missing.
    if "COGSTAT" in df.columns:
        uds_bad = df["NACCUDSD"].isna() | df["NACCUDSD"].isin(list(MISSING_CODES))
        cog_non_dem = df["COGSTAT"].isin([1, 2, 3, 4])  # clinician judged non-demented range
        cnmci_mask = uds_non_dem | (uds_bad & cog_non_dem)
    else:
        cnmci_mask = uds_non_dem

else:
    # This code uses NORMCOG / NACCTMCI If NACCUDSD is not present
    cn_mask  = (df["NORMCOG"] == 1) if "NORMCOG" in df.columns else pd.Series(False, index=df.index)
    mci_mask = df["NACCTMCI"].isin([1,2,3,4]) if "NACCTMCI" in df.columns else pd.Series(False, index=df.index)

    cnmci_mask = cn_mask | mci_mask

    # Fallback for if even those are missing
    if "COGSTAT" in df.columns:
        cog_non_dem = df["COGSTAT"].isin([1,2,3,4])
        cnmci_mask = cnmci_mask | cog_non_dem

baseline_candidates = df[no_dem_mask & cnmci_mask].copy()

baseline = first_row_per_group(baseline_candidates, "NACCID", "visit_date").copy()
baseline = baseline.rename(columns={"visit_date": "baseline_date"})

print("Baseline participants:", baseline["NACCID"].nunique())
baseline[["NACCID","baseline_date"]].head()

Baseline participants: 38341


/var/folders/nq/q3v4dg_s4q78s4w9bbr8zsv40000gn/T/ipykernel_13180/2371402752.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()


,NACCID,baseline_date
0,NACC000011,2006-04-17
1,NACC000034,2015-07-16
2,NACC000067,2012-05-23
3,NACC000073,2016-05-20
4,NACC000138,2023-07-13


In [6]:
def first_row_per_group(df_: pd.DataFrame, group_col: str, sort_col: str) -> pd.DataFrame:
    return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()

if "DEMENTED" in df.columns:
    no_dem_mask = (df["DEMENTED"] == 0)
else:
    no_dem_mask = pd.Series(True, index=df.index)

# Baseline eligibility: NOT demented, and CN/MCI
if "NACCUDSD" in df.columns:
    # Best: UDS-derived cognitive status
    cnmci_mask = df["NACCUDSD"].isin([1, 2, 3])
else:
    # Fallback: use NORMCOG and/or NACCTMCI if present
    cn_mask = (df["NORMCOG"] == 1) if "NORMCOG" in df.columns else pd.Series(False, index=df.index)
    mci_mask = df["NACCTMCI"].isin([1, 2, 3, 4]) if "NACCTMCI" in df.columns else pd.Series(False, index=df.index)

    if ("NORMCOG" in df.columns) or ("NACCTMCI" in df.columns):
        cnmci_mask = cn_mask | mci_mask
    else:
        cnmci_mask = pd.Series(True, index=df.index)

if "COGSTAT" in df.columns:
    cog_ok = ~df["COGSTAT"].isin([0, 9, -4])
    # cnmci_mask = cnmci_mask & cog_ok

baseline_candidates = df[no_dem_mask & cnmci_mask].copy()

baseline = (baseline_candidates
            .sort_values(["NACCID", "visit_date"])
            .groupby("NACCID", as_index=False)
            .first())

baseline["baseline_date"] = baseline["visit_date"]
baseline = baseline.drop(columns=["visit_date"])

print("Baseline participants:", baseline["NACCID"].nunique())
baseline[["NACCID", "baseline_date"]].head()

Baseline participants: 38341


/var/folders/nq/q3v4dg_s4q78s4w9bbr8zsv40000gn/T/ipykernel_13180/136695756.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .first())


,NACCID,baseline_date
0,NACC000011,2006-04-17
1,NACC000034,2015-07-16
2,NACC000067,2012-05-23
3,NACC000073,2016-05-20
4,NACC000138,2023-07-13


## 4) Follow-up time & first dementia event (no horizon restriction yet)
I am first computing time since baseline for every visit, and then identifying each participant’s first post-baseline dementia visit.


In [7]:
def years_between(a: pd.Series, b: pd.Series) -> pd.Series:
    return (b - a).dt.days / 365.25

keep_ids = set(baseline['NACCID'])
df2 = df[df['NACCID'].isin(keep_ids)].merge(
    baseline[['NACCID','baseline_date']], on='NACCID', how='left'
).copy()

df2['t_years'] = years_between(df2['baseline_date'], df2['visit_date'])
post = df2[df2['t_years'] > 0].copy()

print("=== CHECKPOINT: post ===")
print("baseline N:", baseline["NACCID"].nunique())
print("post rows:", len(post), " post unique IDs:", post["NACCID"].nunique())
print("post visit_date min/max:", post["visit_date"].min(), post["visit_date"].max())
print("t_years min/max:", post["t_years"].min(), post["t_years"].max())

max_follow = post.groupby('NACCID')['t_years'].max().rename('max_follow_years')
baseline2 = baseline.merge(max_follow, on='NACCID', how='left')

print("\n=== CHECKPOINT: follow-up ===")
print("max_follow_years describe:")
display(baseline2["max_follow_years"].describe())
print("eligible @4y:", (baseline2["max_follow_years"] >= 4).sum())
print("eligible @5y:", (baseline2["max_follow_years"] >= 5).sum())

baseline2['max_follow_years'] = baseline2['max_follow_years'].fillna(0)

#DEMENTED: 0-No, 1-Yes
#NACCUDSD:1-Normal, 2-Impaired not MCI, 3-MCI, 4-Dementia
if 'NACCUDSD' in post.columns:
    dem_events_all = post[post['NACCUDSD'] == 4].copy()
    dementia_def = 'NACCUDSD==4'
else:
    dem_events_all = post[post['DEMENTED'] == 1].copy()
    dementia_def = 'DEMENTED==1'

first_dem_ever = (dem_events_all.sort_values(['NACCID','visit_date'])
                           .groupby('NACCID', as_index=False)
                           .first()
                           .rename(columns={'visit_date':'dementia_date'}))

first_dem_ever = first_dem_ever.merge(
    baseline2[['NACCID', 'baseline_date', 'max_follow_years']],
    on='NACCID',
    how='left',
    suffixes=('', '_base')   # prevent baseline_date_x/y
)

# Normalizing baseline_date if it got duplicated:
if 'baseline_date' not in first_dem_ever.columns:
    for c in ['baseline_date_base', 'baseline_date_x', 'baseline_date_y']:
        if c in first_dem_ever.columns:
            first_dem_ever['baseline_date'] = first_dem_ever[c]
            break
        
first_dem_ever['time_to_dem_years'] = years_between(first_dem_ever['baseline_date'], first_dem_ever['dementia_date'])

print("\n=== CHECKPOINT: first_dem_ever ===")
print("first_dem_ever unique IDs:", first_dem_ever["NACCID"].nunique())
print("time_to_dem_years min/median/max:",
      first_dem_ever["time_to_dem_years"].min(),
      first_dem_ever["time_to_dem_years"].median(),
      first_dem_ever["time_to_dem_years"].max())

print('Dementia event definition:', dementia_def)
print('Participants with ANY post-baseline dementia event:', first_dem_ever['NACCID'].nunique())
first_dem_ever[['NACCID','baseline_date','dementia_date','time_to_dem_years']].head()

=== CHECKPOINT: post ===
baseline N: 38341
post rows: 123654  post unique IDs: 27703
post visit_date min/max: 2006-04-03 00:00:00 2025-10-21 00:00:00
t_years min/max: 0.008213552361396304 19.392197125256672

=== CHECKPOINT: follow-up ===
max_follow_years describe:


count    27703.000000
mean         5.499126
std          4.268776
min          0.109514
25%          2.080767
50%          4.301164
75%          7.685147
max         19.392197
Name: max_follow_years, dtype: float64

eligible @4y: 14853
eligible @5y: 12474

=== CHECKPOINT: first_dem_ever ===
first_dem_ever unique IDs: 5502
time_to_dem_years min/median/max: 0.4188911704312115 2.6885694729637235 19.304585900068446
Dementia event definition: NACCUDSD==4
Participants with ANY post-baseline dementia event: 5502


/var/folders/nq/q3v4dg_s4q78s4w9bbr8zsv40000gn/T/ipykernel_13180/2987371432.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .first()


,NACCID,baseline_date,dementia_date,time_to_dem_years
0,NACC000034,2015-07-16,2021-05-21,5.848049
1,NACC000162,2009-03-10,2017-11-10,8.670773
2,NACC000225,2009-09-23,2017-07-07,7.786448
3,NACC000271,2007-06-20,2009-05-14,1.900068
4,NACC000333,2021-05-13,2024-07-24,3.197810


In [8]:
HORIZON_YEARS = 5  # or 4

window = post[post["t_years"] <= HORIZON_YEARS].copy()

if "NACCUDSD" in window.columns:
    dem_events_H = window[window["NACCUDSD"] == 4].copy()
else:
    dem_events_H = window[window["DEMENTED"] == 1].copy()

first_dem_within_H = (dem_events_H.sort_values(["NACCID", "visit_date"])
                          .groupby("NACCID", as_index=False)
                          .first()
                          .rename(columns={"visit_date": "dementia_date"}))

# Merging only max follow-up
first_dem_within_H = first_dem_within_H.merge(
    baseline2[["NACCID", "max_follow_years"]],
    on="NACCID", how="left"
)

# Computing time-to-dementia using the baseline_date already in first_dem_within_H
if "baseline_date" not in first_dem_within_H.columns:
    if "baseline_date_x" in first_dem_within_H.columns:
        first_dem_within_H = first_dem_within_H.rename(columns={"baseline_date_x": "baseline_date"})
    elif "baseline_date_y" in first_dem_within_H.columns:
        first_dem_within_H = first_dem_within_H.rename(columns={"baseline_date_y": "baseline_date"})
    else:
        raise KeyError("No baseline_date column found in first_dem_within_H.")

first_dem_within_H["time_to_dem_years"] = years_between(first_dem_within_H["baseline_date"], first_dem_within_H["dementia_date"])

print("Participants with dementia within", HORIZON_YEARS, "y:", first_dem_within_H["NACCID"].nunique())
print("time_to_dem_years min/median/max:",
      first_dem_within_H["time_to_dem_years"].min(),
      first_dem_within_H["time_to_dem_years"].median(),
      first_dem_within_H["time_to_dem_years"].max())

Participants with dementia within 5 y: 3958
time_to_dem_years min/median/max: 0.4188911704312115 2.005475701574264 4.999315537303217


/var/folders/nq/q3v4dg_s4q78s4w9bbr8zsv40000gn/T/ipykernel_13180/2085793406.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .first()


## 5) Etiology subtype mapping (event-level)
Using primary etiology if available:
- PD_LBD primary: `NACCLBDP == 1`
- Vascular primary: `CVDIF == 1`
- AD primary: `NACCALZP == 1` (or fallback `NACCALZD == 1`)


In [9]:

import numpy as np
import pandas as pd


def subtype_primary(row):

    formver = row.get("FORMVER", np.nan)

    # PD / LBD
    if pd.notna(row.get("NACCLBDP")) and int(row["NACCLBDP"]) == 1:
        return "PD_LBD"

    # Vascular
    if formver >= 3:
        if pd.notna(row.get("CVDIF")) and int(row["CVDIF"]) == 1:
            return "Vascular"
    else:
        if pd.notna(row.get("VASCIF")) and int(row["VASCIF"]) == 1:
            return "Vascular"
        if pd.notna(row.get("VASCPSIF")) and int(row["VASCPSIF"]) == 1:
            return "Vascular"

    # AD
    if pd.notna(row.get("NACCALZP")) and int(row["NACCALZP"]) == 1:
        return "AD"
    if pd.notna(row.get("NACCALZD")) and int(row["NACCALZD"]) == 1:
        return "AD"

    return "Other/Unk"



def _as_int(x):
    # safe int conversion for strings/float/NA
    if pd.isna(x):
        return None
    try:
        return int(float(x))
    except Exception:
        return None

def subtype_anymention(row) -> str:
    formver = row.get("FORMVER", np.nan)
    formver = float(formver) if pd.notna(formver) else np.nan

    # PD / LBD (any mention)
    # Etiology present (preferred)
    v = _as_int(row.get("NACCLBDE"))
    if v == 1:
        return "PD_LBD"

    # Vascular (any mention)
    if pd.notna(formver) and formver >= 3:
        # UDS v3+: vascular brain injury etiology present
        v = _as_int(row.get("CVD"))
        if v == 1:
            return "Vascular"
    else:
        # UDS v1.2 / v2: vascular dementia etiology flags (probable/possible)
        v = _as_int(row.get("VASC"))
        if v == 1:
            return "Vascular"
        v = _as_int(row.get("VASCPS"))
        if v == 1:
            return "Vascular"

    # AD (any mention)
    v = _as_int(row.get("NACCALZD"))
    if v == 1:
        return "AD"

    return "Other/Unk"

# Subtype labeling at dementia event (v1/v2/v3 compatible)
# Useing functions subtype_primary() and subtype_anymention() defined above.

def subtype_contributing(row) -> str:
    """Contributing etiology ONLY (used as fallback when primary is Other/Unk)."""
    formver = row.get("FORMVER", np.nan)
    try:
        formver = float(formver) if pd.notna(formver) else np.nan
    except Exception:
        formver = np.nan

    # PD/LBD contributing
    if _as_int(row.get("NACCLBDP")) == 2:
        return "PD_LBD"

    # Vascular contributing
    if pd.notna(formver) and formver >= 3:
        if _as_int(row.get("CVDIF")) == 2:
            return "Vascular"
    else:
        if _as_int(row.get("VASCIF")) == 2:
            return "Vascular"
        if _as_int(row.get("VASCPSIF")) == 2:
            return "Vascular"

    # AD contributing
    if _as_int(row.get("NACCALZP")) == 2:
        return "AD"

    return "Other/Unk"


# Stage 1: Primary
first_dem_ever["subtype_primary"] = first_dem_ever.apply(subtype_primary, axis=1)

# Stage 2: Contributing (computed for all, but used only if primary is Other/Unk)
first_dem_ever["subtype_contrib"] = first_dem_ever.apply(subtype_contributing, axis=1)

# Stage 3: Any-mention
first_dem_ever["subtype_any"] = first_dem_ever.apply(subtype_anymention, axis=1)

# Final label: Primary -> Contributing -> Any-mention
first_dem_ever["subtype_primary_contrib_any"] = np.where(
    first_dem_ever["subtype_primary"].ne("Other/Unk"),
    first_dem_ever["subtype_primary"],
    np.where(
        first_dem_ever["subtype_contrib"].ne("Other/Unk"),
        first_dem_ever["subtype_contrib"],
        first_dem_ever["subtype_any"]
    )
)

# Labeling source
first_dem_ever["subtype_label_source_3tier"] = np.select(
    [
        first_dem_ever["subtype_primary"].ne("Other/Unk"),
        first_dem_ever["subtype_primary"].eq("Other/Unk") & first_dem_ever["subtype_contrib"].ne("Other/Unk"),
        first_dem_ever["subtype_primary"].eq("Other/Unk") & first_dem_ever["subtype_contrib"].eq("Other/Unk") & first_dem_ever["subtype_any"].ne("Other/Unk"),
    ],
    ["primary", "fallback_contributing", "fallback_anymention"],
    default="other/unk"
)

print("3-tier label source counts:")
display(first_dem_ever["subtype_label_source_3tier"].value_counts())

print("\nFinal (Primary -> Contrib -> Any) subtype counts:")
display(first_dem_ever["subtype_primary_contrib_any"].value_counts())

rescued_by_contrib = (
    first_dem_ever["subtype_primary"].eq("Other/Unk")
    & first_dem_ever["subtype_contrib"].ne("Other/Unk")
).sum()

rescued_by_any = (
    first_dem_ever["subtype_primary"].eq("Other/Unk")
    & first_dem_ever["subtype_contrib"].eq("Other/Unk")
    & first_dem_ever["subtype_any"].ne("Other/Unk")
).sum()

print("Rescued by contributing:", rescued_by_contrib)
print("Rescued by any-mention (after contrib):", rescued_by_any)


3-tier label source counts:


subtype_label_source_3tier
primary                  4932
other/unk                 528
fallback_contributing      37
fallback_anymention         5
Name: count, dtype: int64


Final (Primary -> Contrib -> Any) subtype counts:


subtype_primary_contrib_any
AD           4304
Other/Unk     528
PD_LBD        384
Vascular      286
Name: count, dtype: int64

Rescued by contributing: 37
Rescued by any-mention (after contrib): 5


## 6) Horizon sweep tables
For each horizon H:
- how many baseline participants have ≥ H years follow-up
- how many have dementia by H years
- dementia-by-H counts by subtype (primary and any-mention)


In [10]:
# Applying labels on the 'first dementia ever' table
first_dem_ever["subtype_primary"] = first_dem_ever.apply(subtype_primary, axis=1)
first_dem_ever["subtype_any"]     = first_dem_ever.apply(subtype_anymention, axis=1)
first_dem_ever["subtype_contrib"] = first_dem_ever.apply(subtype_contributing, axis=1)

# 3-tier: Primary -> Contributing -> Any-mention
first_dem_ever["subtype_primary_contrib"] = np.where(
    first_dem_ever["subtype_primary"].eq("Other/Unk"),
    first_dem_ever["subtype_contrib"],
    first_dem_ever["subtype_primary"],
)

first_dem_ever["subtype_primary_contrib_any"] = np.where(
    first_dem_ever["subtype_primary_contrib"].eq("Other/Unk"),
    first_dem_ever["subtype_any"],
    first_dem_ever["subtype_primary_contrib"],
)

first_dem_ever["subtype_label_source_3tier"] = "primary"
first_dem_ever.loc[
    first_dem_ever["subtype_primary"].eq("Other/Unk") & ~first_dem_ever["subtype_contrib"].eq("Other/Unk"),
    "subtype_label_source_3tier"
] = "fallback_contributing"
first_dem_ever.loc[
    first_dem_ever["subtype_primary_contrib"].eq("Other/Unk") & ~first_dem_ever["subtype_any"].eq("Other/Unk"),
    "subtype_label_source_3tier"
] = "fallback_anymention"
first_dem_ever.loc[
    first_dem_ever["subtype_primary_contrib_any"].eq("Other/Unk"),
    "subtype_label_source_3tier"
] = "other/unk"

print("3-tier label source counts:")
display(first_dem_ever["subtype_label_source_3tier"].value_counts())

print("\nFinal (Primary -> Contrib -> Any) subtype counts:")
display(first_dem_ever["subtype_primary_contrib_any"].value_counts())


3-tier label source counts:


subtype_label_source_3tier
primary                  4932
other/unk                 528
fallback_contributing      37
fallback_anymention         5
Name: count, dtype: int64


Final (Primary -> Contrib -> Any) subtype counts:


subtype_primary_contrib_any
AD           4304
Other/Unk     528
PD_LBD        384
Vascular      286
Name: count, dtype: int64

In [11]:

import numpy as np
import pandas as pd

# Horizon range
MIN_H = 3
MAX_H = 12

results = []

print("\n=== PRE-SWEEP sanity ===")
print("baseline2 N:", baseline2["NACCID"].nunique())
print("first_dem_ever N:", first_dem_ever["NACCID"].nunique())

# building cohort-style labels for a given horizon
def build_labels_A(H: int) -> pd.Series:
    # dementia within H (event-based; no requirement to have >=H follow-up)
    dem_H = first_dem_ever.loc[first_dem_ever["time_to_dem_years"] <= H, ["NACCID", "subtype_primary_contrib_any"]].copy()

    # Mapping subtype to ID
    subtype_map = dem_H.set_index("NACCID")["subtype_primary_contrib_any"].to_dict()
    dem_ids = set(subtype_map.keys())

    # Follow-up eligibility for non-dementia labels
    has_follow = baseline2.set_index("NACCID")["max_follow_years"].ge(H)

    # Labelling each baseline participant
    labels = []
    for nid, ok in has_follow.items():
        if nid in dem_ids:
            st = subtype_map[nid]
            labels.append(st if st in {"AD", "PD_LBD", "Vascular"} else "MixedOrOther")
        else:
            #labels.append(f"NoDementia_{H}y" if ok else "Censored")
            labels.append(f"NoDementia_Hy" if ok else "Censored")
    return pd.Series(labels, index=has_follow.index, name="label_A")

# only keeping those with >=H follow-up (so no censored)
def build_labels_B(H: int) -> pd.Series:
    eligible = baseline2.loc[baseline2["max_follow_years"].ge(H), "NACCID"]
    eligible_set = set(eligible)

    dem_H = first_dem_ever.loc[
        (first_dem_ever["time_to_dem_years"] <= H) &
        (first_dem_ever["NACCID"].isin(eligible_set)),
        ["NACCID", "subtype_primary_contrib_any"]
    ].copy()

    subtype_map = dem_H.set_index("NACCID")["subtype_primary_contrib_any"].to_dict()
    dem_ids = set(subtype_map.keys())

    labels = []
    for nid in eligible:
        if nid in dem_ids:
            st = subtype_map[nid]
            labels.append(st if st in {"AD", "PD_LBD", "Vascular"} else "MixedOrOther")
        else:
            #labels.append(f"NoDementia_{H}y")
            labels.append(f"NoDementia_Hy")
    return pd.Series(labels, index=eligible, name="label_B")

for H in range(MIN_H, MAX_H + 1):

    # A- Event-only dementia counts (for reference)
    dem_within_H_A = first_dem_ever[first_dem_ever["time_to_dem_years"] <= H]
    counts_A = dem_within_H_A["subtype_primary_contrib_any"].value_counts()

    # B- Strict eligibility dementia counts (for reference)
    eligible_ids = baseline2.loc[baseline2["max_follow_years"] >= H, "NACCID"]
    dem_within_H_B = first_dem_ever[
        (first_dem_ever["time_to_dem_years"] <= H) &
        (first_dem_ever["NACCID"].isin(set(eligible_ids)))
    ]
    counts_B = dem_within_H_B["subtype_primary_contrib_any"].value_counts()

    # Cohort-style label counts (A and B)
    label_A = build_labels_A(H)
    label_B = build_labels_B(H)

    vcA = label_A.value_counts()
    vcB = label_B.value_counts()

    results.append({
        "Horizon": H,

        # Dementia event counts
        "A_Total_DemEvents": len(dem_within_H_A),
        "A_AD": counts_A.get("AD", 0),
        "A_PD_LBD": counts_A.get("PD_LBD", 0),
        "A_Vascular": counts_A.get("Vascular", 0),
        "A_OtherOrUnk": counts_A.get("Other/Unk", 0),

        "B_Total_DemEvents": len(dem_within_H_B),
        "B_AD": counts_B.get("AD", 0),
        "B_PD_LBD": counts_B.get("PD_LBD", 0),
        "B_Vascular": counts_B.get("Vascular", 0),
        "B_OtherOrUnk": counts_B.get("Other/Unk", 0),

        # Eligible cohort size in B
        "B_Eligible_Cohort_Size": len(eligible_ids),

        # Cohort label counts
        f"A_NoDementia_{H}y": vcA.get(f"NoDementia_{H}y", 0),
        f"A_Censored_{H}y": vcA.get("Censored", 0),
        "A_MixedOrOther": vcA.get("MixedOrOther", 0),

        f"B_NoDementia_{H}y": vcB.get(f"NoDementia_{H}y", 0),
        "B_MixedOrOther": vcB.get("MixedOrOther", 0),
    })

horizon_df = pd.DataFrame(results)
display(horizon_df)



=== PRE-SWEEP sanity ===
baseline2 N: 38341
first_dem_ever N: 5502


,Horizon,A_Total_DemEvents,A_AD,A_PD_LBD,A_Vascular,A_OtherOrUnk,B_Total_DemEvents,B_AD,B_PD_LBD,B_Vascular,B_OtherOrUnk,B_Eligible_Cohort_Size,A_NoDementia_3y,A_Censored_3y,A_MixedOrOther,B_NoDementia_3y,B_MixedOrOther,A_NoDementia_4y,A_Censored_4y,B_NoDementia_4y,A_NoDementia_5y,A_Censored_5y,B_NoDementia_5y,A_NoDementia_6y,A_Censored_6y,B_NoDementia_6y,A_NoDementia_7y,A_Censored_7y,B_NoDementia_7y,A_NoDementia_8y,A_Censored_8y,B_NoDementia_8y,A_NoDementia_9y,A_Censored_9y,B_NoDementia_9y,A_NoDementia_10y,A_Censored_10y,B_NoDementia_10y,A_NoDementia_11y,A_Censored_11y,B_NoDementia_11y,A_NoDementia_12y,A_Censored_12y,B_NoDementia_12y
0,3,2929,2272,225,105,327,1598,1274,120,48,156,17724,0.0,19286.0,327,0.0,156,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,3526,2729,269,144,384,1559,1256,102,61,140,14853,NaN,NaN,384,NaN,140,0.0,21521.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5,3958,3068,297,172,421,1391,1140,71,57,123,12474,NaN,NaN,421,NaN,123,NaN,NaN,NaN,0.0,23300.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6,4286,3320,311,200,455,1146,931,47,57,111,10147,NaN,NaN,455,NaN,111,NaN,NaN,NaN,NaN,NaN,NaN,0.0,25054.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7,4550,3538,327,214,471,955,778,42,47,88,8139,NaN,NaN,471,NaN,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,26607.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,8,4737,3681,342,233,481,782,628,34,43,77,6595,NaN,NaN,481,NaN,77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,27791.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,9,4891,3811,352,241,487,631,509,29,31,62,5404,NaN,NaN,487,NaN,62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,28677.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,10,5020,3909,361,255,495,484,380,25,28,51,4412,NaN,NaN,495,NaN,51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,29393.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
8,11,5118,3989,367,260,502,380,300,20,22,38,3582,NaN,NaN,502,NaN,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,30021.0,0.0,NaN,NaN,NaN
9,12,5207,4060,369,270,508,288,231,11,16,30,2831,NaN,NaN,508,NaN,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,30591.0,0.0


# Section 2D: Trying various combination of features as experiment to generate separate cohot files and then use them in modelling code to check the output score

In [12]:
import os
import numpy as np
import pandas as pd


HORIZON_YEARS = 5 
H = HORIZON_YEARS

# 1) Dementia within H (event window)
dem_H = first_dem_ever.loc[
    first_dem_ever["time_to_dem_years"] <= H,
    ["NACCID", "dementia_date", "time_to_dem_years",
     "subtype_primary_contrib_any", "subtype_label_source_3tier"]
].copy()

# 2) Merge onto baseline2 and assign labels (old notebook style)
cohort = baseline2.merge(dem_H, on="NACCID", how="left")

cohort["has_dementia_H"] = cohort["dementia_date"].notna()
cohort["has_H_followup"] = cohort["max_follow_years"] >= H

def assign_label(r):
    if r["has_dementia_H"]:
        st = r.get("subtype_primary_contrib_any", "Other/Unk")
        if st in {"AD", "PD_LBD", "Vascular"}:
            return st
        return "MixedOrOther"
    if r["has_H_followup"]:
        #return f"NoDementia_{H}y"
        return f"NoDementia_Hy"
    return "Censored"

cohort["label"] = cohort.apply(assign_label, axis=1)

print("Cohort rows:", len(cohort))
display(cohort["label"].value_counts(dropna=False))

# 3) Build training export X
STARTER_FEATURES = [
    "NACCID","baseline_date",
    "NACCAGE","SEX","RACE","HISPANIC","EDUC","NACCUDSD",

    # Vascular risk / history (baseline-safe, non-leaky)
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
    "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",

    # Medication proxies for vascular risk treatment (baseline-safe)
    "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD",

    # Other helpful baseline covariates
    "APNEA","DEP2YRS",

    # Cognition / function
    "NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON",
    "CDRSUM","CDRGLOB",

        # PD/LBD symptom block (UDS B9 clinician judgment)
    "COGFLUC",   # fluctuating cognition (0/1/9) :contentReference[oaicite:0]{index=0}
    "BEVHALL",   # visual hallucinations (0/1/9) :contentReference[oaicite:1]{index=1}
    "BEREM",     # REM sleep behavior disorder (0/1/9) :contentReference[oaicite:5]{index=5}
    # Motor / Parkinsonism exam block (UDS B8)
    "PARKSIGN","BRADY","RESTTRL","RESTTRR","RIGIDL","RIGIDR","SLOWINGL","SLOWINGR","PARKGAIT","GAITDIS",
]


cols = [c for c in STARTER_FEATURES if c in baseline.columns]
X = baseline[cols].copy()

# Merging label and follow-up info onto X
X = X.merge(
    cohort[["NACCID","baseline_date","label","max_follow_years","has_H_followup","has_dementia_H"]],
    on=["NACCID","baseline_date"],
    how="left"
)


# 4A) UDS missing-code cleansing

# Common UDS special missing / not-applicable codes
# -4 is the big one; 8/9 often used for NA/Unknown for coded fields.
GLOBAL_MISSING = {-4}

# For many binary/coded clinical vars: 8/9 represent NA/Unknown (dictionary-dependent).
CODED_MISSING = {-4, 8, 9}

NUMERIC_TEST_EXTRA_MISSING = {88, 99, 888, 999}

# 1) Force numeric vars to numeric and replace special codes with NaN
NUM_VARS = ["NACCAGE","EDUC","NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON","CDRSUM","CDRGLOB"]

for v in NUM_VARS:
    if v in X.columns:
        X[v] = pd.to_numeric(X[v], errors="coerce")

        # Always treating -4 as missing
        X.loc[X[v].isin(GLOBAL_MISSING), v] = np.nan

        # For cognitive test-like vars, also treating 88/99/... as missing
        if v in {"NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON","CDRSUM","CDRGLOB"}:
            X.loc[X[v].isin(NUMERIC_TEST_EXTRA_MISSING), v] = np.nan

# 2) For coded/binary medical history + meds vars, replace -4/8/9 with NaN
CODED_VARS = [
    "SEX","RACE","HISPANIC","NACCUDSD",
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
    "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",
    "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD",
    "APNEA","DEP2YRS",
    "COGFLUC","BEVHALL", "BEREM",
    # Motor block
    "PARKSIGN","BRADY","RESTTRL","RESTTRR","RIGIDL","RIGIDR","SLOWINGL","SLOWINGR","PARKGAIT","GAITDIS",
]

for v in CODED_VARS:
    if v in X.columns:
        X[v] = pd.to_numeric(X[v], errors="coerce")
        X.loc[X[v].isin(CODED_MISSING), v] = np.nan

# 3) Adding missing indicators for the coded vars too
for v in CODED_VARS:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)



# 4) Clean coded numerics + add missing indicators
NUM_VARS = ["NACCAGE","EDUC","NACCMMSE","NACCMOCA","TRAILA","TRAILB","CDRSUM","CDRGLOB"]

for v in ["TRAILB","TRAILA","NACCMMSE","NACCMOCA","CDRSUM"]:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)


NACC_CSF_PATH  = os.path.join(DATA_DIR, "investigator_fcsf_nacc72.csv")

if "NACC_CSF_PATH" in globals() and os.path.exists(NACC_CSF_PATH):
    csf = pd.read_csv(NACC_CSF_PATH, low_memory=False)
    csf["NACCID"] = csf["NACCID"].astype(str)

    for c in ["CSFABETA","CSFPTAU","CSFTTAU"]:
        if c in csf.columns:
            csf[f"{c}_missing"] = csf[c].isna().astype(int)

    def make_csf_date(df_: pd.DataFrame) -> pd.Series:
        y = pd.to_numeric(df_.get("CSFLPYR"), errors="coerce")
        m = pd.to_numeric(df_.get("CSFLPMO"), errors="coerce").fillna(6).astype(int).clip(1,12)
        d = pd.to_numeric(df_.get("CSFLPDY"), errors="coerce").fillna(15).astype(int).clip(1,28)
        return pd.to_datetime(dict(year=y, month=m, day=d), errors="coerce")

    csf["csf_date"] = make_csf_date(csf)

    csf = csf.merge(X[["NACCID","baseline_date"]], on="NACCID", how="inner")
    csf = csf.dropna(subset=["csf_date","baseline_date"]).copy()
    csf["abs_days_from_baseline"] = (csf["csf_date"] - csf["baseline_date"]).abs().dt.days

    csf_nearest = csf.sort_values(["NACCID","abs_days_from_baseline"]).groupby("NACCID", as_index=False).first()

    merge_cols = ["NACCID","CSFABETA","CSFPTAU","CSFTTAU","abs_days_from_baseline"]
    merge_cols = [c for c in merge_cols if c in csf_nearest.columns]
    X = X.merge(csf_nearest[merge_cols], on="NACCID", how="left")

    print("Merged CSF. Rows:", len(X))
    if "CSFABETA" in X.columns:
        print("CSFABETA non-null:", X["CSFABETA"].notna().sum())
else:
    print("CSF file not found (or NACC_CSF_PATH not set); skipping CSF merge.")

# Removing unnecessary missing flags added earlier
IMPORTANT_MISSING = [
    "TRAILA",
    "TRAILB",
    "NACCMMSE",
    "NACCMOCA",
    "CDRSUM"
]

for v in IMPORTANT_MISSING:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)

drop_missing_cols = [
    c for c in X.columns
    if c.endswith("_missing") and
       c.replace("_missing","") not in IMPORTANT_MISSING
]

X = X.drop(columns=drop_missing_cols)

# 6) Feature engineering (vascular-focused)

# 6) Feature engineering (clean engineered features)
import numpy as np

def _bin0(df, cols):
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return None
    return df[cols].fillna(0)

# Vascular risk burden
vasc_risk_cols = ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","CVCHF","CVANGINA","CVBYPASS","CVHVALVE"]
B = _bin0(X, vasc_risk_cols)
if B is not None:
    X["VASC_RISK_COUNT"] = B.sum(axis=1)

# Vascular meds burden
vasc_med_cols = ["NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"]
M = _bin0(X, vasc_med_cols)
if M is not None:
    X["VASC_MED_COUNT"] = M.sum(axis=1)

# Stroke/TIA
S = _bin0(X, ["CBSTROKE","CBTIA"])
if S is not None:
    X["STROKE_TIA_ANY"] = (S.sum(axis=1) > 0).astype(int)

# Executive ratio (TrailB / (TrailA+1))
if "TRAILB" in X.columns and "TRAILA" in X.columns:
    X["EXEC_RATIO_TRAILB_TRAILA"] = X["TRAILB"] / (X["TRAILA"] + 1.0)

# Gait
G = _bin0(X, ["GAITDIS"])
if G is not None:
    X["GAIT_ANY"] = (G.sum(axis=1) > 0).astype(int)

print("Engineered feats present:",
      [c for c in ["VASC_RISK_COUNT","VASC_MED_COUNT","STROKE_TIA_ANY","EXEC_RATIO_TRAILB_TRAILA","GAIT_ANY"] if c in X.columns])

# 7) Save outputs
#out_full = f"cohort_full_H{H}.csv"   # full baseline2 + label + event info (debug)
out_XH   = f"cohort_H{H}.csv"        # training export for this horizon
out_X    = "cohort.csv"            

#out_full_path = os.path.join(OUT_DIR, out_full)
out_XH_path = os.path.join(OUT_DIR, out_XH)
out_X_path = os.path.join(OUT_DIR, out_X)

# cohort.to_csv(out_full_path, index=False)
X.to_csv(out_XH_path, index=False)
X.to_csv(out_X_path, index=False)

# print("Wrote:", out_full, out_XH, "and", out_X)
print("Training export label counts:")
display(X["label"].value_counts(dropna=False))



# 7) Save outputs (single + per-experiment files)
import json

BASE_COLS = [
    "NACCID", "baseline_date",
    "label", "max_follow_years", "has_H_followup", "has_dementia_H"
]

# Defining experiment feature groups
DEMOGRAPHIC = ["NACCAGE","SEX","RACE","HISPANIC","EDUC","NACCUDSD"]

# Cognitive + function (+ missing flags)
COGNITIVE = [
    "NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON",
    "CDRSUM","CDRGLOB",
    "NACCMMSE_missing","NACCMOCA_missing","TRAILA_missing","TRAILB_missing","CDRSUM_missing"
]

# Medical / vascular block
MEDICAL_VASCULAR = [
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA",
    "CVCHF","CVANGINA","CVBYPASS","CVANGIO","CVHVALVE","CVOTHR",
    "APNEA","DEP2YRS",
    # meds / treatment proxies
    "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"
]

# Motor / PD-LBD exam + symptom block
MOTOR_PD = [
    "COGFLUC","BEVHALL","BEREM",
    "PARKSIGN","BRADY","RESTTRL","RESTTRR","RIGIDL","RIGIDR",
    "SLOWINGL","SLOWINGR","PARKGAIT","GAITDIS"
]

# CSF biomarkers
CSF = ["CSFABETA","CSFPTAU","CSFTTAU","abs_days_from_baseline"]  
VASC_ENGINEERED = ["VASC_RISK_COUNT", "STROKE_TIA_ANY", "VASC_MED_COUNT", "GAIT_ANY"]
EXEC_ENGINEERED = ["EXEC_RATIO_TRAILB_TRAILA"]

EXPERIMENTS = {
    "E0_demo": DEMOGRAPHIC,
    "E1_demo_cog": DEMOGRAPHIC + COGNITIVE,
    "E2_demo_cog_med": DEMOGRAPHIC + COGNITIVE + MEDICAL_VASCULAR,
    "E3_demo_cog_motor": DEMOGRAPHIC + COGNITIVE + MOTOR_PD,
    "E4_full_clinical": DEMOGRAPHIC + COGNITIVE + MEDICAL_VASCULAR + MOTOR_PD,
    # Below are some of the CSF subset experiments
    "E5_full_plus_csf": DEMOGRAPHIC + COGNITIVE + MEDICAL_VASCULAR + MOTOR_PD + CSF,
    "E6_demo_csf": DEMOGRAPHIC + CSF,
    "E7_demo_med": DEMOGRAPHIC + MEDICAL_VASCULAR,
    # E8 cleaning vascular
    "E8_clean_vascular": DEMOGRAPHIC + COGNITIVE + MOTOR_PD + VASC_ENGINEERED,

    # E9 cleaning exec ratio: use ratio, drop raw Trails later
    "E9_clean_exec_ratio": DEMOGRAPHIC + COGNITIVE + MEDICAL_VASCULAR + MOTOR_PD + EXEC_ENGINEERED,

    # E10 executive-memory axis: will use engineered EXEC_MINUS_MEM + MEM_SCORE
    "E10_exec_mem_axis_raw": DEMOGRAPHIC + COGNITIVE + MEDICAL_VASCULAR + MOTOR_PD,
}

# Select columns safely 
def select_cols(df_, cols):
    cols = list(dict.fromkeys(cols))  # de-dupe, preserve order
    present = [c for c in cols if c in df_.columns]
    missing = [c for c in cols if c not in df_.columns]
    return present, missing

out_XH   = f"cohort_H{H}.csv"
out_X    = "cohort.csv"

out_XH_path = os.path.join(OUT_DIR, out_XH)
out_X_path  = os.path.join(OUT_DIR, out_X)

X.to_csv(out_XH_path, index=False)
X.to_csv(out_X_path, index=False)

print("Wrote:", out_XH, "and", out_X)
print("Training export label counts:")
display(X["label"].value_counts(dropna=False))

manifest_rows = []
exp_dir = OUT_DIR 

for exp_name, feat_list in EXPERIMENTS.items():
    wanted = BASE_COLS + feat_list
    present, missing = select_cols(X, wanted)

    # keeping only columns that exist
    X_exp = X[present].copy()

    # saving file
    exp_file = f"cohort_{exp_name}_H{H}.csv"
    exp_path = os.path.join(exp_dir, exp_file)
    X_exp.to_csv(exp_path, index=False)

    manifest_rows.append({
        "experiment": exp_name,
        "file": exp_file,
        "n_rows": len(X_exp),
        "n_cols": X_exp.shape[1],
        "missing_cols": ";".join(missing) if missing else "",
        "features_json": json.dumps(feat_list),
    })

    if missing:
        print(f"[{exp_name}] wrote {exp_file} (NOTE: missing {len(missing)} cols: {missing[:8]}{'...' if len(missing)>8 else ''})")
    else:
        print(f"[{exp_name}] wrote {exp_file}")

manifest = pd.DataFrame(manifest_rows)
manifest_file = f"cohort_manifest_H{H}.csv"
manifest_path = os.path.join(exp_dir, manifest_file)
manifest.to_csv(manifest_path, index=False)
print("Wrote manifest:", manifest_file)

display(manifest[["experiment","file","n_rows","n_cols","missing_cols"]])


Cohort rows: 38341


label
Censored         23300
NoDementia_Hy    11083
AD                3068
MixedOrOther       421
PD_LBD             297
Vascular           172
Name: count, dtype: int64

Merged CSF. Rows: 38341
CSFABETA non-null: 1559
Engineered feats present: ['VASC_RISK_COUNT', 'VASC_MED_COUNT', 'STROKE_TIA_ANY', 'EXEC_RATIO_TRAILB_TRAILA', 'GAIT_ANY']
Training export label counts:


label
Censored         23300
NoDementia_Hy    11083
AD                3068
MixedOrOther       421
PD_LBD             297
Vascular           172
Name: count, dtype: int64

Wrote: cohort_H5.csv and cohort.csv
Training export label counts:


label
Censored         23300
NoDementia_Hy    11083
AD                3068
MixedOrOther       421
PD_LBD             297
Vascular           172
Name: count, dtype: int64

[E0_demo] wrote cohort_E0_demo_H5.csv
[E1_demo_cog] wrote cohort_E1_demo_cog_H5.csv
[E2_demo_cog_med] wrote cohort_E2_demo_cog_med_H5.csv
[E3_demo_cog_motor] wrote cohort_E3_demo_cog_motor_H5.csv
[E4_full_clinical] wrote cohort_E4_full_clinical_H5.csv
[E5_full_plus_csf] wrote cohort_E5_full_plus_csf_H5.csv
[E6_demo_csf] wrote cohort_E6_demo_csf_H5.csv
[E7_demo_med] wrote cohort_E7_demo_med_H5.csv
[E8_clean_vascular] wrote cohort_E8_clean_vascular_H5.csv
[E9_clean_exec_ratio] wrote cohort_E9_clean_exec_ratio_H5.csv
[E10_exec_mem_axis_raw] wrote cohort_E10_exec_mem_axis_raw_H5.csv
Wrote manifest: cohort_manifest_H5.csv


,experiment,file,n_rows,n_cols,missing_cols
0,E0_demo,cohort_E0_demo_H5.csv,38341,12,
1,E1_demo_cog,cohort_E1_demo_cog_H5.csv,38341,25,
2,E2_demo_cog_med,cohort_E2_demo_cog_med_H5.csv,38341,46,
3,E3_demo_cog_motor,cohort_E3_demo_cog_motor_H5.csv,38341,38,
4,E4_full_clinical,cohort_E4_full_clinical_H5.csv,38341,59,
5,E5_full_plus_csf,cohort_E5_full_plus_csf_H5.csv,38341,63,
6,E6_demo_csf,cohort_E6_demo_csf_H5.csv,38341,16,
7,E7_demo_med,cohort_E7_demo_med_H5.csv,38341,33,
8,E8_clean_vascular,cohort_E8_clean_vascular_H5.csv,38341,42,
9,E9_clean_exec_ratio,cohort_E9_clean_exec_ratio_H5.csv,38341,60,


3.1 Vascular Composite Engineering (E8)
Raw vascular risk variables were replaced with:
•	VASC_RISK_COUNT
•	VASC_MED_COUNT
•	STROKE_TIA_ANY
This reduced multicollinearity and improved balanced accuracy slightly (~+0.002–0.003).
Conclusion:
Aggregated vascular burden provided modest improvement but did not dramatically improve subtype separability.
________________________________________
3.2 Executive Dysfunction Ratio (E9)
Created:
EXEC_RATIO_TRAILB_TRAILA
Raw Trails scores removed.
Result:
•	No improvement in balanced accuracy.
•	Slight decrease in vascular recall.
Conclusion:
Executive dysfunction ratio did not meaningfully improve subtype discrimination.
________________________________________
3.3 Executive-Memory Axis (E10)
Constructed:
EXEC_MINUS_MEM
Using train-only z-scored:
•	Trail B (executive)
•	MMSE/MOCA (memory)
Result:
•	Performance similar to baseline.
•	No meaningful vascular improvement.
Conclusion:
Linear combination of executive-memory features did not enhance subtype separability.
________________________________________
3.4 Addition of CSF Biomarkers
CSFABETA, CSFPTAU, CSFTTAU added to E8.
Result:
•	Balanced accuracy ~0.632 (no material change)
•	No vascular recall improvement
Conclusion:
CSF biomarkers did not improve multi-class subtype discrimination at baseline, likely due to overlapping biomarker profiles and limited vascular coverage.


In [13]:
print("Number of modeling columns:", len(X.columns))
print(sorted(X.columns))

Number of modeling columns: 68
['ANIMALS', 'APNEA', 'BEREM', 'BEVHALL', 'BOSTON', 'BRADY', 'CBSTROKE', 'CBTIA', 'CDRGLOB', 'CDRSUM', 'CDRSUM_missing', 'COGFLUC', 'CSFABETA', 'CSFPTAU', 'CSFTTAU', 'CVAFIB', 'CVANGINA', 'CVANGIO', 'CVBYPASS', 'CVCHF', 'CVHVALVE', 'CVOTHR', 'DEP2YRS', 'DIABETES', 'EDUC', 'EXEC_RATIO_TRAILB_TRAILA', 'GAITDIS', 'GAIT_ANY', 'HISPANIC', 'HYPERCHO', 'HYPERTEN', 'NACCAC', 'NACCAGE', 'NACCANGI', 'NACCBETA', 'NACCCCBS', 'NACCDIUR', 'NACCID', 'NACCLIPL', 'NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'NACCUDSD', 'NACCVASD', 'PARKGAIT', 'PARKSIGN', 'RACE', 'RESTTRL', 'RESTTRR', 'RIGIDL', 'RIGIDR', 'SEX', 'SLOWINGL', 'SLOWINGR', 'STROKE_TIA_ANY', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'VASC_MED_COUNT', 'VASC_RISK_COUNT', 'abs_days_from_baseline', 'baseline_date', 'has_H_followup', 'has_dementia_H', 'label', 'max_follow_years']
